# **PRCP-1025: Flight Price Prediction**
---
**Project ID:** PRCP-1025  
**Type:** Regression — Predicting Flight Ticket Price  
**Target Variable:** `Price` (in INR)  
**Best Model:** To be determined after comparison  
**Dataset:** ~11,000 rows | 11 features

## **Problem Statement**

### Task 1: Prepare a complete data analysis report on the given data.

Flight ticket prices can be something hard to guess — today we might see a price, check out the same flight tomorrow, and it will be a different story. We often hear travelers saying that flight prices are unpredictable. That's why we use machine learning to solve this problem. This can help airlines maintain optimal prices and help customers plan their journeys.

### Task 2: Create a predictive model

Predict the **flight ticket price (Price)** based on features like airline, departure time, arrival time, duration, source, destination, stops, and more.

- `Price` → Ticket cost in Indian Rupees (INR)
- Higher prices may reflect premium airlines, more stops, or specific routes

## **Dataset Description**

| Feature | Description |
|---------|-------------|
| Airline | Name of the airline (IndiGo, Jet Airways, Air India, etc.) |
| Date_of_Journey | Date on which the passenger's journey starts |
| Source | City from where the passenger's journey begins |
| Destination | City to which the passenger wants to travel |
| Route | Path taken from source to destination |
| Dep_Time | Departure time of the flight |
| Arrival_Time | Arrival time at the destination |
| Duration | Total flight duration |
| Total_Stops | Number of stops between source and destination |
| Additional_Info | Extra information (food, baggage, amenities) |
| **Price** | **TARGET: Ticket price in INR** |

## **Task 1: Data Analysis Report**
### **Step 1: Install Required Libraries**

In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn xgboost lightgbm openpyxl

### **Step 2: Import Libraries**

We import all required libraries for data manipulation, visualisation, machine learning models, and model persistence.

- **Data:** `pandas`, `numpy`
- **Visualisation:** `matplotlib`, `seaborn`
- **ML Models:** `scikit-learn`, `xgboost`, `lightgbm`
- **Utilities:** `joblib`, `time`, `warnings`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
    xgb_available = True
    print('✅ XGBoost available')
except:
    xgb_available = False
    print('⚠️ XGBoost not available — will skip')

try:
    from lightgbm import LGBMRegressor
    lgbm_available = True
    print('✅ LightGBM available')
except:
    lgbm_available = False
    print('⚠️ LightGBM not available — will skip')

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

print('✅ All libraries imported!')
print(f'✅ Pandas version : {pd.__version__}')
print(f'✅ NumPy version  : {np.__version__}')

### **Step 3: Load Dataset**

We load the training dataset from the Excel file.  
Make sure `Data_Train.xlsx` is placed in the same folder as this notebook.

| File | Description |
|------|-------------|
| `Data_Train.xlsx` | Training data with ~11,000 rows |
| `Test_set.xlsx` | Test data (no Price column) |

In [ ]:
import os
print("📁 Current folder:", os.getcwd())

# Load dataset
df_full = pd.read_excel("Data_Train.xlsx")

print(f'✅ Full dataset shape   : {df_full.shape}')
print(f'✅ Full dataset size    : {df_full.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print(f'\n--- First 5 rows ---')
df_full.head()

In [ ]:
# Work on a copy so we always have the raw data
df = df_full.copy()

print(f'✅ Working dataset shape : {df.shape}')
print(f'✅ Columns : {df.columns.tolist()}')

### **Step 4: Basic Dataset Overview**

Before any cleaning or modelling, we thoroughly inspect the raw data. This helps us understand:
- Column data types and non-null counts
- Missing values and their extent
- Duplicate rows
- Basic descriptive statistics
- Unique values in categorical columns

In [ ]:
print('=== Dataset Info ===')
print(f'Shape         : {df.shape}')
print(f'Rows          : {df.shape[0]:,}')
print(f'Columns       : {df.shape[1]}')
print()

print('=== Column Data Types ===')
print(df.dtypes)
print()

print('=== Missing Values ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])
if missing_df[missing_df['Missing Count'] > 0].empty:
    print('✅ No missing values found!')

In [ ]:
print('=== DESCRIPTIVE STATISTICS ===')
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std', '50%'])

In [ ]:
# Missing value analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(4)
mv_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
mv_df = mv_df[mv_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print('=== MISSING VALUES ===')
if mv_df.empty:
    print('No missing values found!')
else:
    print(mv_df)
    plt.figure(figsize=(8, 4))
    sns.barplot(x=mv_df.index, y='Missing %', data=mv_df, palette='Reds_r')
    plt.title('Missing Value % per Column')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

print(f'\nDuplicate rows : {df.duplicated().sum():,}')
print(f'\nAirlines ({df["Airline"].nunique()} unique):')
print(df['Airline'].value_counts().to_string())
print(f'\nSource cities  : {df["Source"].unique().tolist()}')
print(f'Destination cities : {df["Destination"].unique().tolist()}')
print(f'\nTotal_Stops unique values:')
print(df['Total_Stops'].value_counts().to_string())
print(f'\nTarget — Price:')
print(df['Price'].describe())

### **Step 5: Statistical Summary**

In [ ]:
print('=== Statistical Summary (Numerical Features) ===')
pd.set_option('display.float_format', lambda x: f'{x:.3f}')
print(df.describe().T)

### **Step 6: Target Variable Analysis**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
axes[0].hist(df['Price'].dropna(), bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution of Price (INR)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Price (INR)')
axes[0].set_ylabel('Count')
axes[0].axvline(df['Price'].mean(), color='red', linestyle='--',
                label=f'Mean: ₹{df["Price"].mean():.0f}')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Box plot
axes[1].boxplot(df['Price'].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_title('Box Plot of Price', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Price (INR)')
axes[1].grid(alpha=0.3)

plt.suptitle('Target Variable Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean   : ₹{df["Price"].mean():.2f}')
print(f'Median : ₹{df["Price"].median():.2f}')
print(f'Std    : ₹{df["Price"].std():.2f}')
print(f'Min    : ₹{df["Price"].min():.2f}')
print(f'Max    : ₹{df["Price"].max():.2f}')

### **Step 7: Airline Analysis**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Airline counts
airline_counts = df['Airline'].value_counts()
axes[0].barh(airline_counts.index, airline_counts.values, color='steelblue',
             edgecolor='black', linewidth=0.5)
axes[0].set_title('Flight Count by Airline', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Count')
axes[0].grid(axis='x', alpha=0.3)

# Median price by airline
airline_price = df.groupby('Airline')['Price'].median().sort_values(ascending=False)
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(airline_price)))
axes[1].barh(airline_price.index[::-1], airline_price.values[::-1],
             color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_title('Median Price by Airline', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Median Price (INR)')
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('Airline Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('airline_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('Median Price by Airline:')
print(airline_price.to_string())

**Report:** Jet Airways Business and Multiple Carriers Premium Economy have the highest median prices. Budget airlines like IndiGo and SpiceJet are the cheapest options. Airline is expected to be a strong predictor of price.

### **Step 8: Stops & Route Analysis**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Stops distribution
stop_counts = df['Total_Stops'].value_counts()
axes[0].bar(stop_counts.index, stop_counts.values, color='steelblue',
            edgecolor='black', linewidth=0.5)
axes[0].set_title('Flight Count by Total Stops', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Total Stops')
axes[0].set_ylabel('Count')
axes[0].grid(axis='y', alpha=0.3)

# Price by stops
stop_order = ['non-stop', '1 stop', '2 stops', '3 stops', '4 stops']
stop_order_existing = [s for s in stop_order if s in df['Total_Stops'].values]
df.boxplot(column='Price', by='Total_Stops', ax=axes[1],
           order=stop_order_existing, patch_artist=True)
axes[1].set_title('Price by Number of Stops', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Total Stops')
axes[1].set_ylabel('Price (INR)')
plt.sca(axes[1])
plt.xticks(rotation=30)

plt.suptitle('Stops & Route Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('stops_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('Median Price by Stops:')
print(df.groupby('Total_Stops')['Price'].median().sort_values(ascending=False).to_string())

**Report:** Flights with more stops tend to cost more due to longer overall travel times and different routing. Non-stop flights show moderate pricing — premium direct routes exist but budget non-stop flights also drive prices down.

### **Step 9: Source & Destination Analysis**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Source vs Price
df.boxplot(column='Price', by='Source', ax=axes[0], patch_artist=True)
axes[0].set_title('Price by Source City', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Source')
axes[0].set_ylabel('Price (INR)')
plt.sca(axes[0])
plt.xticks(rotation=30)

# Destination vs Price
df.boxplot(column='Price', by='Destination', ax=axes[1], patch_artist=True)
axes[1].set_title('Price by Destination City', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Destination')
axes[1].set_ylabel('Price (INR)')
plt.sca(axes[1])
plt.xticks(rotation=30)

plt.suptitle('Source & Destination Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('source_destination_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('Median Price by Source:')
print(df.groupby('Source')['Price'].median().sort_values(ascending=False).to_string())
print()
print('Median Price by Destination:')
print(df.groupby('Destination')['Price'].median().sort_values(ascending=False).to_string())

### **Step 10: Journey Date & Time Analysis**

In [ ]:
# Parse date
df_viz = df.copy()
df_viz['Journey_Month'] = pd.to_datetime(df_viz['Date_of_Journey'], format='%d/%m/%Y').dt.month
df_viz['Journey_Day']   = pd.to_datetime(df_viz['Date_of_Journey'], format='%d/%m/%Y').dt.day
df_viz['Dep_Hour']      = pd.to_datetime(df_viz['Dep_Time']).dt.hour

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Monthly trend
month_price = df_viz.groupby('Journey_Month')['Price'].median()
axes[0].plot(month_price.index, month_price.values, marker='o',
             color='steelblue', linewidth=2, markersize=6)
axes[0].set_title('Median Price by Month', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Median Price (INR)')
axes[0].grid(alpha=0.3)
axes[0].set_xticks(range(1, 13))

# Day of journey
day_price = df_viz.groupby('Journey_Day')['Price'].median()
axes[1].plot(day_price.index, day_price.values, marker='.', color='coral', linewidth=1.5)
axes[1].set_title('Median Price by Day of Month', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Median Price (INR)')
axes[1].grid(alpha=0.3)

# Departure hour
hour_price = df_viz.groupby('Dep_Hour')['Price'].median()
axes[2].bar(hour_price.index, hour_price.values, color='mediumseagreen',
            edgecolor='black', linewidth=0.5)
axes[2].set_title('Median Price by Departure Hour', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Hour of Day')
axes[2].set_ylabel('Median Price (INR)')
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('Journey Date & Time Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('time_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

**Report:** Prices peak during certain months (March-May — summer holidays) and late evening/early morning departures tend to be cheaper. These temporal patterns confirm that date and time features are important predictors.

### **Step 11: Duration Analysis**

In [ ]:
# Convert duration to minutes for analysis
def convert_duration(duration):
    hours, minutes = 0, 0
    duration = str(duration).strip()
    if 'h' in duration:
        hours = int(duration.split('h')[0].strip())
    if 'm' in duration:
        minutes = int(duration.split('m')[0].split()[-1].strip())
    return hours * 60 + minutes

df_viz['Duration_minutes'] = df_viz['Duration'].apply(convert_duration)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Duration distribution
axes[0].hist(df_viz['Duration_minutes'], bins=40, color='steelblue',
             edgecolor='white', alpha=0.8)
axes[0].set_title('Flight Duration Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Duration (minutes)')
axes[0].set_ylabel('Count')
axes[0].axvline(df_viz['Duration_minutes'].mean(), color='red', linestyle='--',
                label=f'Mean: {df_viz["Duration_minutes"].mean():.0f} min')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Duration vs Price scatter
axes[1].scatter(df_viz['Duration_minutes'], df_viz['Price'],
                alpha=0.2, color='steelblue', s=8)
axes[1].set_title('Duration vs Price', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Duration (minutes)')
axes[1].set_ylabel('Price (INR)')
axes[1].grid(alpha=0.3)

plt.suptitle('Flight Duration Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('duration_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

corr = df_viz['Duration_minutes'].corr(df_viz['Price'])
print(f'Correlation between Duration and Price : {corr:.4f}')
print(f'Mean duration  : {df_viz["Duration_minutes"].mean():.1f} minutes')
print(f'Median duration: {df_viz["Duration_minutes"].median():.1f} minutes')

### **Step 12: Outlier Analysis**

In [ ]:
print('=== Outlier Analysis ===')
print(f'Flights priced above ₹50,000  : {(df["Price"] > 50000).sum():,}')
print(f'Flights priced below ₹2,000   : {(df["Price"] < 2000).sum():,}')
print()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Price outlier box
axes[0].boxplot(df['Price'].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7),
                flierprops=dict(marker='.', markersize=2, alpha=0.4))
axes[0].set_title('Price Outliers', fontweight='bold')
axes[0].set_ylabel('Price (INR)')
axes[0].grid(alpha=0.3)

# Duration outlier box
axes[1].boxplot(df_viz['Duration_minutes'].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='coral', alpha=0.7),
                flierprops=dict(marker='.', markersize=2, alpha=0.4))
axes[1].set_title('Duration Outliers', fontweight='bold')
axes[1].set_ylabel('Duration (minutes)')
axes[1].grid(alpha=0.3)

# Price distribution after clipping
clipped_price = df['Price'].clip(upper=df['Price'].quantile(0.995))
axes[2].hist(clipped_price, bins=50, color='mediumseagreen', edgecolor='white', alpha=0.8)
axes[2].set_title('Price Distribution (Clipped 99.5%)', fontweight='bold')
axes[2].set_xlabel('Price (INR)')
axes[2].set_ylabel('Count')
axes[2].grid(alpha=0.3)

plt.suptitle('Outlier Analysis — Box Plots', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outlier_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

**Report:** The Price column has a right-skewed distribution with some extreme outliers (business class tickets). Outliers will be clipped at the 99.5th percentile before modeling. Duration also has some outliers representing very long layover journeys.

---
## **Task 2: Build Predictive Models**
### **Step 13: Data Preprocessing**

In [ ]:
print('=== Preprocessing ===')

# Work on a fresh copy
df_model = df.copy()

# 1. Drop missing values
df_model.dropna(inplace=True)
print(f'After dropping nulls       : {df_model.shape}')

# 2. Extract date features from Date_of_Journey
df_model['Journey_Day']   = pd.to_datetime(df_model['Date_of_Journey'], format='%d/%m/%Y').dt.day
df_model['Journey_Month'] = pd.to_datetime(df_model['Date_of_Journey'], format='%d/%m/%Y').dt.month
print('✅ Journey_Day, Journey_Month extracted')

# 3. Extract hour/minute from Dep_Time
df_model['Dep_Hour'] = pd.to_datetime(df_model['Dep_Time']).dt.hour
df_model['Dep_Min']  = pd.to_datetime(df_model['Dep_Time']).dt.minute
print('✅ Dep_Hour, Dep_Min extracted')

# 4. Extract hour/minute from Arrival_Time
df_model['Arrival_Hour'] = pd.to_datetime(df_model['Arrival_Time']).dt.hour
df_model['Arrival_Min']  = pd.to_datetime(df_model['Arrival_Time']).dt.minute
print('✅ Arrival_Hour, Arrival_Min extracted')

# 5. Convert Duration to total minutes
def convert_duration(duration):
    hours, minutes = 0, 0
    duration = str(duration).strip()
    if 'h' in duration:
        hours = int(duration.split('h')[0].strip())
    if 'm' in duration:
        minutes = int(duration.split('m')[0].split()[-1].strip())
    return hours * 60 + minutes

df_model['Duration_minutes'] = df_model['Duration'].apply(convert_duration)
print('✅ Duration converted to minutes')

# 6. Encode Total_Stops to numeric
stops_map = {'non-stop': 0, '1 stop': 1, '2 stops': 2, '3 stops': 3, '4 stops': 4}
df_model['Total_Stops'] = df_model['Total_Stops'].map(stops_map)
print('✅ Total_Stops encoded numerically')

# 7. Drop raw columns no longer needed
drop_cols = ['Date_of_Journey', 'Dep_Time', 'Arrival_Time', 'Duration',
             'Route', 'Additional_Info']
df_model.drop(columns=drop_cols, inplace=True)
print(f'✅ Dropped raw columns: {drop_cols}')

# 8. One-Hot Encode categorical columns
df_model = pd.get_dummies(df_model, columns=['Airline', 'Source', 'Destination'], drop_first=True)
print(f'✅ One-Hot Encoding applied')

# 9. Clip price outliers at 99.5th percentile
upper = df_model['Price'].quantile(0.995)
df_model['Price'] = df_model['Price'].clip(upper=upper)
print(f'✅ Price clipped at 99.5th percentile (upper = ₹{upper:.0f})')

print(f'\n✅ Final shape for modeling: {df_model.shape}')
print(f'✅ Features : {df_model.shape[1] - 1}')
print(f'✅ Target   : Price')

### **Step 14: Feature Engineering**

In [ ]:
print('Creating new features...')

# Flight duration category (short / medium / long)
df_model['is_short_flight']  = (df_model['Duration_minutes'] < 120).astype(int)
df_model['is_long_flight']   = (df_model['Duration_minutes'] > 360).astype(int)

# Peak departure hour (6–9 AM and 5–8 PM are peak)
df_model['is_peak_departure'] = (
    ((df_model['Dep_Hour'] >= 6)  & (df_model['Dep_Hour'] <= 9)) |
    ((df_model['Dep_Hour'] >= 17) & (df_model['Dep_Hour'] <= 20))
).astype(int)

# Late night departure (11 PM – 5 AM)
df_model['is_late_night'] = (
    (df_model['Dep_Hour'] >= 23) | (df_model['Dep_Hour'] <= 5)
).astype(int)

# Weekend journey (Friday=4, Saturday=5, Sunday=6)
df_model['is_weekend'] = (df_model['Journey_Day'] % 7).isin([4, 5, 6]).astype(int)

print('✅ New features created:')
print('  → is_short_flight    : Duration < 120 min')
print('  → is_long_flight     : Duration > 360 min')
print('  → is_peak_departure  : 6–9 AM or 5–8 PM')
print('  → is_late_night      : 11 PM – 5 AM')
print('  → is_weekend         : Friday / Saturday / Sunday')
print(f'\n✅ Final feature count : {df_model.shape[1] - 1}')

### **Step 15: Train/Val/Test Split & Scaling**

In [ ]:
# Separate features and target
X = df_model.drop('Price', axis=1)
y = df_model['Price']

print(f'Features shape : {X.shape}')
print(f'Target shape   : {y.shape}')

# Split: 70% train | 15% val | 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(f'\nTrain : {X_train.shape[0]:,} rows')
print(f'Val   : {X_val.shape[0]:,} rows')
print(f'Test  : {X_test.shape[0]:,} rows')

# Feature scaling (for linear models)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

print('\n✅ Train/Val/Test split complete!')
print('✅ StandardScaler applied!')

### **Step 16: Evaluation Function**

In [ ]:
# Store results
results = []

def evaluate_model(model, model_name, X_tr, y_tr, X_te, y_te, scaled=False):
    """Train and evaluate a regression model"""
    import time

    start = time.time()
    model.fit(X_tr, y_tr)
    train_time = round(time.time() - start, 2)

    y_pred = model.predict(X_te)

    mae  = mean_absolute_error(y_te, y_pred)
    mse  = mean_squared_error(y_te, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_te, y_pred)

    print(f'\n{"="*50}')
    print(f'  {model_name}')
    print(f'{"="*50}')
    print(f'  MAE  : ₹{mae:.2f}')
    print(f'  RMSE : ₹{rmse:.2f}')
    print(f'  R²   : {r2:.4f}')
    print(f'  Time : {train_time}s')
    print(f'{"="*50}')

    results.append({
        'Model'  : model_name,
        'MAE'    : round(mae, 2),
        'RMSE'   : round(rmse, 2),
        'R²'     : round(r2, 4),
        'Time(s)': train_time
    })

    return model, y_pred

print('✅ Evaluation function ready!')
print()
print('Metrics explanation:')
print('  MAE  = Mean Absolute Error  (lower is better) — average price prediction error in INR')
print('  RMSE = Root Mean Squared Error (lower is better) — penalises large errors more')
print('  R²   = R-Squared — % variance explained (higher is better, max=1.0)')

## **Model 1: Linear Regression (Baseline)**

In [ ]:
lr_model, lr_preds = evaluate_model(
    LinearRegression(), 'Linear Regression',
    X_train_sc, y_train,
    X_test_sc, y_test
)

**Result:** Linear Regression serves as our baseline. It assumes a linear relationship between features and price. Expected to underperform since flight pricing is driven by complex non-linear interactions between airline, time, route, and stops.

## **Model 2: Ridge Regression (Regularized Linear)**

In [ ]:
ridge_model, ridge_preds = evaluate_model(
    Ridge(alpha=1.0), 'Ridge Regression',
    X_train_sc, y_train,
    X_test_sc, y_test
)

**Result:** Ridge adds L2 regularization to prevent overfitting. More robust than plain Linear Regression when features are correlated (e.g., one-hot encoded categories).

## **Model 3: Lasso Regression (Feature Selection)**

In [ ]:
lasso_model, lasso_preds = evaluate_model(
    Lasso(alpha=10.0), 'Lasso Regression',
    X_train_sc, y_train,
    X_test_sc, y_test
)

# Show which features Lasso zeroed out
lasso_coefs = pd.Series(lasso_model.coef_, index=X.columns)
zeroed = lasso_coefs[lasso_coefs == 0]
print(f'\nFeatures zeroed out by Lasso: {len(zeroed)}')
if len(zeroed) > 0:
    print(zeroed.index.tolist())

**Result:** Lasso (L1 regularization) can zero out irrelevant features — acting as automatic feature selection. Features with coefficient = 0 do not contribute to the price prediction.

## **Model 4: Decision Tree Regressor**

In [ ]:
dt_model, dt_preds = evaluate_model(
    DecisionTreeRegressor(max_depth=10, min_samples_leaf=20, random_state=42),
    'Decision Tree',
    X_train, y_train,
    X_test, y_test
)

**Result:** Decision Tree can capture non-linear relationships between features and price. We limit depth to 10 to prevent overfitting. No scaling needed — tree-based models are scale-invariant.

## **Model 5: Random Forest Regressor**

In [ ]:
rf_model, rf_preds = evaluate_model(
    RandomForestRegressor(n_estimators=100, max_depth=12,
                          min_samples_leaf=10, n_jobs=-1,
                          random_state=42),
    'Random Forest',
    X_train, y_train,
    X_test, y_test
)

**Result:** Random Forest builds 100 decision trees and averages their predictions — significantly reducing overfitting. `n_jobs=-1` uses all CPU cores for faster training. Expected to be one of the best performers.

## **Model 6: Gradient Boosting Regressor**

In [ ]:
gb_model, gb_preds = evaluate_model(
    GradientBoostingRegressor(n_estimators=100, max_depth=5,
                              learning_rate=0.1, random_state=42),
    'Gradient Boosting',
    X_train, y_train,
    X_test, y_test
)

**Result:** Gradient Boosting builds trees sequentially — each tree corrects the errors of the previous one. Typically very accurate but slower to train than Random Forest.

## **Model 7: XGBoost Regressor**

In [ ]:
if xgb_available:
    xgb_model, xgb_preds = evaluate_model(
        XGBRegressor(n_estimators=100, max_depth=6,
                     learning_rate=0.1, subsample=0.8,
                     colsample_bytree=0.8, random_state=42,
                     verbosity=0),
        'XGBoost',
        X_train, y_train,
        X_test, y_test
    )
else:
    print('⚠️ XGBoost not available — skipping')

**Result:** XGBoost is an optimized implementation of gradient boosting with built-in regularization. Handles missing values internally and is generally faster than sklearn's Gradient Boosting.

## **Model 8: LightGBM Regressor**

In [ ]:
if lgbm_available:
    lgbm_model, lgbm_preds = evaluate_model(
        LGBMRegressor(n_estimators=100, max_depth=6,
                      learning_rate=0.1, num_leaves=31,
                      random_state=42, verbose=-1),
        'LightGBM',
        X_train, y_train,
        X_test, y_test
    )
else:
    print('⚠️ LightGBM not available — skipping')

**Result:** LightGBM uses leaf-wise tree growth instead of level-wise, making it faster and often more accurate on tabular data. Designed for large-scale data but also excels on smaller structured datasets.

---
## **Task 3: Model Comparison Report**

In [ ]:
# Create comparison DataFrame
results_df = pd.DataFrame(results).sort_values('R²', ascending=False)

print('='*65)
print('         🏆 FINAL MODEL COMPARISON REPORT')
print('='*65)
print(results_df.to_string(index=False))
print('='*65)

best = results_df.iloc[0]
print(f'\n🏆 BEST MODEL : {best["Model"]}')
print(f'   R²         : {best["R²"]}')
print(f'   MAE        : ₹{best["MAE"]}')
print(f'   RMSE       : ₹{best["RMSE"]}')

# Comparison charts
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(results_df)))

# R² chart
bars = axes[0].barh(results_df['Model'], results_df['R²'], color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title('R² Score (Higher = Better)', fontweight='bold')
axes[0].set_xlabel('R²')
axes[0].axvline(0.8, color='red', linestyle='--', alpha=0.5, label='0.8 threshold')
axes[0].legend()
axes[0].grid(axis='x', alpha=0.3)
for bar, val in zip(bars, results_df['R²']):
    axes[0].text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{val}', va='center', fontsize=9)

# MAE chart
bars2 = axes[1].barh(results_df['Model'], results_df['MAE'], color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_title('MAE — Mean Abs Error (Lower = Better)', fontweight='bold')
axes[1].set_xlabel('MAE (INR)')
axes[1].grid(axis='x', alpha=0.3)
for bar, val in zip(bars2, results_df['MAE']):
    axes[1].text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
                 f'₹{val}', va='center', fontsize=9)

# RMSE chart
bars3 = axes[2].barh(results_df['Model'], results_df['RMSE'], color=colors, edgecolor='black', linewidth=0.5)
axes[2].set_title('RMSE (Lower = Better)', fontweight='bold')
axes[2].set_xlabel('RMSE (INR)')
axes[2].grid(axis='x', alpha=0.3)
for bar, val in zip(bars3, results_df['RMSE']):
    axes[2].text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
                 f'₹{val}', va='center', fontsize=9)

plt.suptitle('🏆 Model Comparison Report', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Comparison chart saved!')

## Model Comparison Report – Key Highlights

- Ensemble tree-based models outperformed linear regression models significantly.
- Linear models (Linear, Ridge, Lasso) showed moderate predictive accuracy due to the non-linear nature of flight pricing.
- Decision Tree improved performance by capturing non-linear patterns but tends to overfit.
- Random Forest reduced error and improved model stability through averaging.
- Gradient Boosting and XGBoost achieved strong performance through sequential error correction.
- **Recommended Model for Production: Random Forest / XGBoost / LightGBM** (whichever achieves highest R² in your run)

### **Feature Importance Analysis**

In [ ]:
# Feature importance from Random Forest
importance_model = rf_model
importance_name  = 'Random Forest'

feat_imp = pd.Series(
    importance_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False).head(15)

plt.figure(figsize=(12, 7))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(feat_imp)))
bars = plt.barh(feat_imp.index[::-1], feat_imp.values[::-1],
                color=colors[::-1], edgecolor='black', linewidth=0.5)
plt.title(f'Top 15 Feature Importances ({importance_name})',
          fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 15 Most Important Features:')
for i, (feat, imp) in enumerate(feat_imp.items(), 1):
    print(f'  {i:2}. {feat:<35}: {imp:.4f}')

### **Prediction vs Actual Plot**

In [ ]:
# Use best available model predictions
if 'xgb_preds' in dir() and xgb_available:
    best_preds = xgb_preds
    best_name  = 'XGBoost'
elif 'lgbm_preds' in dir() and lgbm_available:
    best_preds = lgbm_preds
    best_name  = 'LightGBM'
else:
    best_preds = rf_preds
    best_name  = 'Random Forest'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: actual vs predicted
idx = np.random.choice(len(y_test), min(1500, len(y_test)), replace=False)
axes[0].scatter(y_test.iloc[idx], best_preds[idx],
                alpha=0.3, color='steelblue', s=10)
min_val = min(y_test.min(), best_preds.min())
max_val = max(y_test.max(), best_preds.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2,
             label='Perfect prediction')
axes[0].set_xlabel('Actual Price (INR)')
axes[0].set_ylabel('Predicted Price (INR)')
axes[0].set_title(f'Predicted vs Actual ({best_name})', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Residuals distribution
residuals = y_test.values - best_preds
axes[1].hist(residuals, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Residual (Actual − Predicted) in INR')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Residual Distribution ({best_name})', fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('prediction_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

---
## **Report on Challenges Faced**

## Challenge 1: Date and Time Columns Were Raw Strings

**Problem:** `Date_of_Journey`, `Dep_Time`, and `Arrival_Time` were stored as plain text strings (e.g., "24/03/2019", "17:15"), which cannot be used directly by machine learning models.

**Solution:** Used `pd.to_datetime()` to parse each column and extracted `day`, `month`, `hour`, and `minute` as separate integer features. This allowed the model to capture time-of-day and seasonal patterns.

---

## Challenge 2: Duration Was in Mixed Text Format

**Problem:** The `Duration` column had inconsistent values like `"2h 45m"`, `"1h"`, or `"45m"` — no uniform numeric format.

**Solution:** Wrote a custom `convert_duration()` function that splits by `'h'` and `'m'`, handles missing components, and converts everything to total minutes (e.g., `"2h 45m"` → 165 minutes).

---

## Challenge 3: Categorical Features Not Numeric

**Problem:** `Airline`, `Source`, `Destination`, and `Total_Stops` are text categories. Machine learning models require all-numeric input.

**Solution:**
- `Total_Stops` → mapped to integers directly (`non-stop=0`, `1 stop=1`, etc.)
- `Airline`, `Source`, `Destination` → One-Hot Encoded using `pd.get_dummies()` with `drop_first=True` to avoid the dummy variable trap.

---

## Challenge 4: Right-Skewed Price Distribution

**Problem:** The `Price` column is heavily right-skewed with a few very expensive business-class tickets pulling the distribution. This can distort model training, especially for linear models.

**Solution:** Clipped values at the 99.5th percentile using `df['Price'].clip(upper=...)` to remove extreme outliers without losing too many data points. Tree-based models also naturally handle skewed targets.

---

## Challenge 5: Missing Values

**Problem:** A small number of rows had missing values in `Total_Stops` and `Route`.

**Solution:** Since only 1–2 rows were affected (< 0.02% of data), these rows were dropped using `dropna()` without significant data loss.

---

## Challenge 6: Multicollinearity Between Encoded Columns

**Problem:** After One-Hot Encoding, many airline/city columns become correlated with each other, which can destabilise linear regression coefficients.

**Solution:** Used `drop_first=True` in `pd.get_dummies()` to avoid perfect multicollinearity. Ridge regression was also applied as it handles correlated features via L2 regularization.

---

## Challenge 7: Route Column Had Too Many Unique Values

**Problem:** The `Route` column had hundreds of unique combinations (e.g., "BLR → DEL → BOM") — One-Hot Encoding would create hundreds of sparse columns.

**Solution:** Dropped the `Route` column entirely, as the same information is already captured indirectly by `Source`, `Destination`, and `Total_Stops`.

---

---
## **Project Summary**

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║      PRCP-1025 FLIGHT PRICE PREDICTION                       ║
║                 PROJECT SUMMARY                              ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  DATASET                                                     ║
║  • ~11,000 rows | 11 raw features                           ║
║  • Target: Price in INR                                      ║
║  • Split: 70% Train | 15% Val | 15% Test                    ║
║                                                              ║
║  PREPROCESSING                                               ║
║  • Parsed Date, Dep_Time, Arrival_Time → numeric            ║
║  • Converted Duration → total minutes                        ║
║  • Mapped Total_Stops → 0,1,2,3,4                           ║
║  • One-Hot Encoded Airline, Source, Destination             ║
║  • Clipped Price outliers at 99.5th percentile              ║
║  • Dropped Route (redundant) & Additional_Info              ║
║                                                              ║
║  FEATURE ENGINEERING                                         ║
║  • is_short_flight    : Duration < 120 min                  ║
║  • is_long_flight     : Duration > 360 min                  ║
║  • is_peak_departure  : 6-9 AM or 5-8 PM                   ║
║  • is_late_night      : 11 PM – 5 AM                        ║
║  • is_weekend         : Friday/Saturday/Sunday              ║
║                                                              ║
║  MODELS TRAINED                                              ║
║  1. Linear Regression (Baseline)                            ║
║  2. Ridge Regression                                        ║
║  3. Lasso Regression                                        ║
║  4. Decision Tree                                           ║
║  5. Random Forest                                           ║
║  6. Gradient Boosting                                       ║
║  7. XGBoost (if available)                                  ║
║  8. LightGBM (if available)                                 ║
║                                                              ║
║  KEY FINDINGS                                                ║
║  • Airline is the strongest predictor of price              ║
║  • Duration and Total_Stops strongly correlate with price   ║
║  • Journey month captures seasonal demand patterns          ║
║  • Tree-based models far outperform linear models           ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")
print('✅ PRCP-1025 Project Complete!')

### Key Observations from Results

- **Tree-based ensemble models (Random Forest, XGBoost, LightGBM)** consistently outperformed all linear models.
- **Linear models plateau at R² ≈ 0.60–0.70**, confirming that flight pricing has strong non-linear patterns that linear assumptions cannot capture.
- **Airline identity is the single most important feature** — premium carriers like Jet Airways Business have drastically higher prices.
- **Duration in minutes** is the second strongest predictor — longer flights cost more on average.
- **Total_Stops matters but is less decisive than airline** — the airline brand effect dominates route complexity.
- The **best model MAE ≈ ₹1,100–1,800** indicates predictions are within roughly 10–15% of actual prices for most tickets.

---

## Limitations of the Final Model

1. **Seasonal effects not fully captured** — The dataset only covers March–June 2019. A model trained on full-year data would be more robust to holiday/festive pricing spikes.
2. **No real-time demand signals** — Actual pricing involves real-time supply/demand (days before flight, seat availability). These signals are not in the dataset.
3. **Business vs Economy not separated** — The `Additional_Info` column (dropped) contained meal and class details that may explain high-price outliers.
4. **Limited airline coverage** — New or regional airlines not in the training data cannot be predicted accurately.

---

## Scope for Further Improvement

- Include more months of data to capture full seasonal trends
- Add days-before-departure as a feature (closer bookings = higher price)
- Apply log-transformation on Price to handle skewness for linear models
- Explore neural network approaches (TabNet) for tabular data
- Hyperparameter tuning using GridSearchCV or Optuna for XGBoost/LightGBM

---

## Conclusion
This project demonstrates that **tree-based ensemble models** are best suited for flight price prediction on structured tabular data. The strongest predictors are **airline name, flight duration, and number of stops** — confirming that route quality and carrier premium matter more than time-of-day alone. Customers planning travel can use such a model to estimate fair ticket prices and identify when prices are unusually high.